# RoadGuard AI - training

Runs on a free GPU and does everything: downloads the datasets, merges them, trains YOLOv8 and evaluates it on the held-out test split.

**Kaggle:** Settings → Accelerator → GPU T4 x2, and Settings → Internet → On (needs a phone-verified account).
Paste your Roboflow key in step 1, then click **Save Version → Save & Run All**. It runs in the background,
so you can close the laptop. When it finishes, open the version and download `results.zip` from its Output tab.

**Colab:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.

In [ ]:
import os, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn the GPU on first")
if not os.path.exists("roadguard-ai"):
    subprocess.run(["git", "clone", "-q", "https://github.com/Sahil-u07/roadguard-ai"], check=True)
os.chdir("roadguard-ai")
subprocess.run(["pip", "install", "-q", "ultralytics", "roboflow"], check=True)

## 1. Datasets

- Humps/Bumps/Potholes and Indian Roads come from Roboflow Universe. They need a free Roboflow API key
  (roboflow.com → sign up → Settings → API Keys). With no key, these two are skipped.
- Crack-seg (Ultralytics) and a pothole dataset (github.com/michelpf/dataset-pothole) download without a key.

In [ ]:
ROBOFLOW_API_KEY = ""  # paste your key between the quotes

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for workspace, project, folder in [("detection-system", "humps-bumps-potholes-detection", "raw/HumpsBumpsPotholes"),
                                       ("object-detection-dp5wa", "yolo-v8-indian-roads-dataset", "raw/IndianRoads")]:
        p = rf.workspace(workspace).project(project)
        latest = max(p.versions(), key=lambda v: int(v.version))
        latest.download("yolov8", location=folder)
else:
    print("No Roboflow key: skipping Humps/Bumps/Potholes and Indian Roads (no speed bumps in this run)")

In [ ]:
import random, shutil, urllib.request, zipfile
from pathlib import Path

# Crack-seg ships as images/<split> and labels/<split>; merge_datasets.py wants <split>/images and <split>/labels.
urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip", "crack-seg.zip")
zipfile.ZipFile("crack-seg.zip").extractall("raw/_crack")
for split in ("train", "val", "test"):
    for kind in ("images", "labels"):
        shutil.copytree(f"raw/_crack/{kind}/{split}", f"raw/CrackSeg/{split}/{kind}", dirs_exist_ok=True)
Path("raw/CrackSeg/data.yaml").write_text("names: [crack]\n")

# The pothole dataset has train/ and test/ with images and labels side by side and Roboflow-augmented copies
# ("12_jpg.rf.<hash>.jpg"). Valid takes 15% of the train photos, grouped by photo so copies never straddle splits.
subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/michelpf/dataset-pothole", "raw/_pothole"], check=True)
src, dst = Path("raw/_pothole/dataset"), Path("raw/Potholes")
photo = lambda p: p.name.split(".rf.")[0]
train = sorted((src / "train").glob("*.jpg"))
photos = sorted({photo(p) for p in train})
random.Random(0).shuffle(photos)
valid = set(photos[: int(len(photos) * 0.15)])
splits = {"train": [p for p in train if photo(p) not in valid], "valid": [p for p in train if photo(p) in valid],
          "test": sorted((src / "test").glob("*.jpg"))}
for split, images in splits.items():
    for kind in ("images", "labels"):
        (dst / split / kind).mkdir(parents=True, exist_ok=True)
    for img in images:
        shutil.copy2(img, dst / split / "images" / img.name)
        shutil.copy2(img.with_suffix(".txt"), dst / split / "labels" / (img.stem + ".txt"))
(dst / "data.yaml").write_text("names: [pothole]\n")
print({k: len(v) for k, v in splits.items()})

## 2. Map every dataset's classes to our four classes

Each dataset names its classes differently. The rules below map them by keyword (RDD codes included:
D00/D10 cracks, D20 alligator crack, D40 pothole). Anything that matches no rule (cars, manholes...) is dropped.
**Read the printed map** and fix `RULES` if something landed in the wrong class.

In [ ]:
import yaml

ALL_CLASSES = ["Pothole", "Crack", "Severe-Crack", "Speed-Bump"]
RULES = [("severe", "Severe-Crack"), ("alligator", "Severe-Crack"), ("d20", "Severe-Crack"),
         ("pothole", "Pothole"), ("d40", "Pothole"),
         ("crack", "Crack"), ("d00", "Crack"), ("d10", "Crack"),
         ("bump", "Speed-Bump"), ("hump", "Speed-Bump"), ("speed", "Speed-Bump"), ("breaker", "Speed-Bump")]

def unify(name):
    return next((c for key, c in RULES if key in name.lower()), None)

sources = []
for folder in ["raw/HumpsBumpsPotholes", "raw/IndianRoads", "raw/CrackSeg", "raw/Potholes"]:
    if not Path(folder, "data.yaml").exists():
        continue
    names = yaml.safe_load(Path(folder, "data.yaml").read_text())["names"]
    names = names.values() if isinstance(names, dict) else names
    mapping = {n: unify(n) for n in names}
    print(folder, mapping)
    sources.append({"name": Path(folder).name.lower(), "path": folder, "map": mapping})
# Keep only classes some dataset provides, so the app never lists a class the model was not trained on.
CLASSES = [c for c in ALL_CLASSES if any(c in s["map"].values() for s in sources)]
print("Classes:", CLASSES)
Path("sources_auto.yaml").write_text(yaml.safe_dump({"classes": CLASSES, "seed": 0, "sources": sources}, sort_keys=False))

In [ ]:
r = subprocess.run(["python", "merge_datasets.py", "sources_auto.yaml", "datasets/roadguard"], capture_output=True, text=True)
print(r.stdout, r.stderr)
r.check_returncode()

## 3. Train

Starts from COCO-pretrained YOLOv8s (transfer learning). `time=10` stops training cleanly after 10 hours,
before Kaggle's 12-hour limit, and keeps the best model so far. `patience=15` stops early when the validation
score stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(data="datasets/roadguard/data.yaml", epochs=60, imgsz=640, batch=16, patience=15, time=10,
            project="runs", name="roadguard", exist_ok=True, seed=0)
best_path = model.trainer.best  # Ultralytics picks the run folder, so ask it where best.pt went
print(best_path)

## 4. Evaluate on the test split

The test split is never used for training or for choosing the best epoch, so these are the numbers to report.

In [ ]:
import json

best = YOLO(best_path)
m = best.val(data="datasets/roadguard/data.yaml", split="test", project="runs", name="test", exist_ok=True)

p, r = m.box.mp, m.box.mr
metrics = {"precision": p, "recall": r, "f1": 2 * p * r / (p + r), "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "per_class": {best.names[int(c)]: dict(zip(["precision", "recall", "mAP50", "mAP50-95"], m.box.class_result(i)))
                         for i, c in enumerate(m.box.ap_class_index)}}
print(json.dumps(metrics, indent=2, default=float))
json.dump(metrics, open(m.save_dir / "metrics.json", "w"), indent=2, default=float)
test_dir = m.save_dir

## 5. Save the results

`results.zip` holds `best.pt`, the merge summary inputs, the training curves and the test results.
Kaggle: download it from the version's Output tab. Colab: run `from google.colab import files; files.download("results.zip")`.

In [ ]:
Path("weights").mkdir(exist_ok=True)
shutil.copy(best_path, "weights/best.pt")
shutil.copy("sources_auto.yaml", test_dir / "sources_auto.yaml")
with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (model.trainer.save_dir, test_dir):
        for f in Path(folder).rglob("*"):
            z.write(f)
    z.write("weights/best.pt")
shutil.copy("results.zip", "..")  # Kaggle lists /kaggle/working in the Output tab
print("results.zip ready")